In [36]:
import math, time
from collections import defaultdict

def build_problem(rows, cols, demand, tower_configs, metric="manhattan"):
    def in_range(t, c, radius):
        dr, dc = abs(t[0]-c[0]), abs(t[1]-c[1])
        if metric == "manhattan":  return dr + dc <= radius
        if metric == "euclidean":  return math.hypot(dr, dc) <= radius
        if metric == "chebyshev":  return max(dr, dc) <= radius

    cells = [(r, c) for r in range(rows) for c in range(cols)]
    domains = {}
    covers = defaultdict(set)

    for cell in cells:
        d = demand[cell[0]][cell[1]]
        dom = set()
        for ci, (radius, cap, cost) in enumerate(tower_configs):
            if cap < d:
                continue
            for pos in cells:
                if in_range(pos, cell, radius):
                    dom.add((pos, ci))
                    covers[(pos, ci)].add(cell)
        domains[cell] = dom

    # prune dominated towers
    keys = list(covers.keys())
    dominated = set()
    for i, a in enumerate(keys):
        if not covers[a]:
            dominated.add(a)
            continue
        for j, b in enumerate(keys):
            if i == j or b in dominated:
                continue
            if (covers[a] <= covers[b] and
                tower_configs[a[1]][2] >= tower_configs[b[1]][2]):
                dominated.add(a)
                break
    for d in dominated:
        del covers[d]

    valid = set(covers.keys())
    for cell in cells:
        domains[cell] = domains[cell] & valid

    return cells, domains, covers


In [15]:
def forward_check(cell, tower_key, domains, remaining_cap, demand, unassigned):
    """
    After assigning cell to tower_key, prune domains of unassigned cells.
    Returns: pruned list [(cell, removed_values)] for undo, or None if wipeout.
    """
    pos, ci = tower_key
    pruned = []

    for other in unassigned:
        if other == cell:
            continue
        if tower_key not in domains[other]:
            continue
        # check if tower can still handle this cell's demand
        if remaining_cap[tower_key] < demand[other[0]][other[1]]:
            domains[other].discard(tower_key)
            pruned.append((other, tower_key))
            if len(domains[other]) == 0:
                return None  # domain wipeout — backtrack
    return pruned

In [16]:
from collections import deque

def ac3(domains, remaining_cap, demand, covers, unassigned):
    """
    Enforce arc consistency. Two cells sharing a candidate tower are 'neighbors'.
    Returns: pruned list for undo, or None if wipeout.
    """
    pruned = []

    # build arcs: (cell_i, cell_j) where they share at least one tower
    queue = deque()
    unassigned_set = set(unassigned)
    # only check unassigned cells
    tower_to_cells = defaultdict(set)
    for cell in unassigned_set:
        for tk in domains[cell]:
            tower_to_cells[tk].add(cell)

    for tk, cell_set in tower_to_cells.items():
        cell_list = list(cell_set)
        for i in range(len(cell_list)):
            for j in range(i+1, len(cell_list)):
                queue.append((cell_list[i], cell_list[j], tk))
                queue.append((cell_list[j], cell_list[i], tk))

    while queue:
        ci, cj, tk = queue.popleft()
        if ci not in unassigned_set or cj not in unassigned_set:
            continue
        if tk not in domains[ci]:
            continue

        # check: if ci uses tower tk, can cj still find a valid assignment?
        di = demand[ci[0]][ci[1]]
        dj = demand[cj[0]][cj[1]]

        # if both assigned to same tower, need cap >= di + dj
        # if cj has other options, it's fine
        cap_if_both = remaining_cap[tk] - di - dj
        if cap_if_both < 0:
            # cj can't also use tk if ci uses tk
            # but does cj have OTHER options?
            other_options = domains[cj] - {tk}
            if len(other_options) == 0:
                # ci using tk would wipe cj — remove tk from ci
                domains[ci].discard(tk)
                pruned.append((ci, tk))
                if len(domains[ci]) == 0:
                    return None
                # re-enqueue arcs involving ci
                for tk2 in domains[ci]:
                    for neighbor in tower_to_cells.get(tk2, set()):
                        if neighbor != ci and neighbor in unassigned_set:
                            queue.append((neighbor, ci, tk2))

    return pruned

In [17]:
def select_mrv(domains, unassigned):
    """Pick unassigned cell with smallest domain (fail-fast)."""
    return min(unassigned, key=lambda c: len(domains[c]))

def order_lcv(cell, domains, remaining_cap, demand, unassigned, covers):
    """Order domain values by least constraining: fewest domain reductions."""
    def impact(tower_key):
        pos, ci = tower_key
        d = demand[cell[0]][cell[1]]
        new_cap = remaining_cap[tower_key] - d
        count = 0
        for other in unassigned:
            if other == cell:
                continue
            if tower_key in domains[other]:
                if new_cap < demand[other[0]][other[1]]:
                    count += 1  # would lose this option
        return count

    return sorted(domains[cell], key=impact)

In [37]:


def solve(rows, cols, demand, tower_configs, metric="manhattan", time_limit=30):
    cells, domains, covers = build_problem(rows, cols, demand, tower_configs, metric)

    remaining_cap = {}
    for tk in covers:
        remaining_cap[tk] = tower_configs[tk[1]][1]

    assignment = {}
    best = {"cost": float("inf"), "solution": None}
    start = time.time()

    def knapsack_greedy(cells_list, cap, reverse=True):
        cells_list = sorted(cells_list, key=lambda c: demand[c[0]][c[1]], reverse=reverse)
        subset, used = [], 0
        for c in cells_list:
            d = demand[c[0]][c[1]]
            if used + d <= cap:
                subset.append(c)
                used += d
        return subset if subset else None

    def get_subsets(tower, uncovered, candidates):
        reachable = [c for c in covers[tower] if c in uncovered]
        cap = remaining_cap[tower]
        if not reachable:
            return []
        subsets = []

        s1 = knapsack_greedy(reachable, cap, reverse=True)
        if s1: subsets.append(s1)

        s2 = knapsack_greedy(reachable, cap, reverse=False)
        if s2 and set(s2) != set(s1 or []): subsets.append(s2)

        options = {c: sum(1 for tk in candidates if c in covers.get(tk, set()))
                   for c in reachable}
        desperate = sorted(reachable, key=lambda c: options[c])
        s3 = knapsack_greedy(desperate, cap, reverse=True)
        if s3 and set(s3) not in [set(s) for s in subsets]: subsets.append(s3)

        return subsets

    def select_tower(candidates, uncovered):
        cell_options = {c: 0 for c in uncovered}
        for tk in candidates:
            for c in covers.get(tk, set()):
                if c in cell_options:
                    cell_options[c] += 1

        def score(tk):
            return sum(1.0 / max(cell_options[c], 1)
                       for c in covers[tk] if c in cell_options)

        return max(candidates, key=score)

    def backtrack(uncovered, cost_so_far, candidates):
        if time.time() - start > time_limit:
            return
        if not uncovered:
            if cost_so_far < best["cost"]:
                best["cost"] = cost_so_far
                best["solution"] = dict(assignment)
            return
        if cost_so_far >= best["cost"]:
            return
        for c in uncovered:
            if not any(c in covers.get(tk, set()) for tk in candidates):
                return

        tower = select_tower(candidates, uncovered)
        pos, ci = tower

        for subset in get_subsets(tower, uncovered, candidates):
            cap_used = sum(demand[c[0]][c[1]] for c in subset)
            for c in subset:
                assignment[c] = tower
            remaining_cap[tower] -= cap_used

            backtrack(uncovered - set(subset),
                      cost_so_far + tower_configs[ci][2],
                      candidates - {tower})

            remaining_cap[tower] += cap_used
            for c in subset:
                del assignment[c]

        backtrack(uncovered, cost_so_far, candidates - {tower})

    candidates = set(covers.keys())
    backtrack(set(cells), 0, candidates)
    return best["solution"], best["cost"]

In [38]:
# Test 1: 1x1 trivial
solution, cost = solve(1, 1, [[3]], [(1, 5, 8)])
print(f"1x1: cost={cost}, solution={solution}")

# Test 2: 2x2 uniform demand
solution, cost = solve(2, 2,
    [[2, 2],
     [2, 2]],
    [(1, 8, 10), (2, 16, 25)])
print(f"2x2 uniform: cost={cost}")

# Test 3: 3x3 with one high-demand cell
solution, cost = solve(3, 3,
    [[1, 1, 1],
     [1, 9, 1],
     [1, 1, 1]],
    [(1, 5, 8), (2, 12, 20), (1, 10, 15)])
print(f"3x3 spike: cost={cost}")

# Test 4: 4x4 gradient demand
solution, cost = solve(4, 4,
    [[1, 2, 3, 4],
     [2, 3, 4, 5],
     [3, 4, 5, 6],
     [4, 5, 6, 7]],
    [(1, 8, 10), (2, 20, 25), (3, 40, 50)])
print(f"4x4 gradient: cost={cost}")

# Test 5: 5x5 random-ish
solution, cost = solve(5, 5,
    [[3, 1, 4, 1, 5],
     [9, 2, 6, 5, 3],
     [5, 8, 9, 7, 9],
     [3, 2, 3, 8, 4],
     [6, 2, 6, 4, 3]],
    [(1, 10, 12), (2, 20, 28), (3, 35, 45)])
print(f"5x5: cost={cost}")

1x1: cost=8, solution={(0, 0): ((0, 0), 0)}
2x2 uniform: cost=20
3x3 spike: cost=35
4x4 gradient: cost=85
5x5: cost=154


In [39]:
def print_solution(solution, demand, tower_configs):
    if not solution:
        print("No feasible solution")
        return
    towers = {}
    for cell, tk in solution.items():
        towers.setdefault(tk, []).append(cell)
    for (pos, ci), served in towers.items():
        r, cap, cst = tower_configs[ci]
        total_d = sum(demand[c[0]][c[1]] for c in served)
        print(f"  Tower {pos} cfg={ci} R={r} C={cap} $={cst} | "
              f"serves {len(served)} cells, load={total_d}/{cap}")

In [40]:
import pygame, math

CELL = 60
FONT_SIZE = 14
PAD = 180  # right panel for legend

CONFIG_COLORS = [
    (231, 76, 60),
    (52, 152, 219),
    (46, 204, 113),
    (241, 196, 15),
    (155, 89, 182),
]

def run_viz(rows, cols, demand, solution, tower_configs, metric="manhattan"):
    if not solution:
        print("No solution to visualize")
        return

    pygame.init()
    W = cols * CELL + PAD
    H = max(rows * CELL, 300)
    screen = pygame.display.set_mode((W, H))
    pygame.display.set_caption("Tower Placement — Capacity Budget CSP")
    font = pygame.font.SysFont("monospace", FONT_SIZE)
    font_sm = pygame.font.SysFont("monospace", 11)

    # group solution by tower
    towers = {}
    for cell, tk in solution.items():
        towers.setdefault(tk, []).append(cell)

    # assign color per tower
    tower_list = list(towers.keys())
    tower_color = {tk: CONFIG_COLORS[i % len(CONFIG_COLORS)] for i, tk in enumerate(tower_list)}

    def in_range(t, c, radius):
        dr, dc = abs(t[0]-c[0]), abs(t[1]-c[1])
        if metric == "manhattan":  return dr + dc <= radius
        if metric == "euclidean":  return math.hypot(dr, dc) <= radius
        if metric == "chebyshev":  return max(dr, dc) <= radius

    running = True
    while running:
        for ev in pygame.event.get():
            if ev.type == pygame.QUIT: running = False
            if ev.type == pygame.KEYDOWN and ev.key == pygame.K_ESCAPE: running = False

        screen.fill((20, 20, 25))

        # draw cell backgrounds colored by assigned tower
        for cell, tk in solution.items():
            r, c = cell
            color = tower_color[tk]
            dim = (color[0]//5, color[1]//5, color[2]//5)
            pygame.draw.rect(screen, dim, (c*CELL, r*CELL, CELL, CELL))

        # grid lines
        for r in range(rows + 1):
            pygame.draw.line(screen, (50, 50, 50), (0, r*CELL), (cols*CELL, r*CELL))
        for c in range(cols + 1):
            pygame.draw.line(screen, (50, 50, 50), (c*CELL, 0), (c*CELL, rows*CELL))

        # demand values
        for r in range(rows):
            for c in range(cols):
                txt = font.render(str(demand[r][c]), True, (180, 180, 180))
                screen.blit(txt, (c*CELL + 4, r*CELL + 2))

        # assignment lines: cell center -> tower center
        for tk, served in towers.items():
            pos, ci = tk
            tx, ty = pos[1]*CELL + CELL//2, pos[0]*CELL + CELL//2
            color = tower_color[tk]
            for cell in served:
                cx, cy = cell[1]*CELL + CELL//2, cell[0]*CELL + CELL//2
                if (cx, cy) != (tx, ty):
                    line_surf = pygame.Surface((cols*CELL, rows*CELL), pygame.SRCALPHA)
                    pygame.draw.line(line_surf, (*color, 80), (tx, ty), (cx, cy), 1)
                    screen.blit(line_surf, (0, 0))

        # range circles (translucent)
        for tk in tower_list:
            pos, ci = tk
            radius = tower_configs[ci][0]
            cx, cy = pos[1]*CELL + CELL//2, pos[0]*CELL + CELL//2
            range_surf = pygame.Surface((cols*CELL, rows*CELL), pygame.SRCALPHA)
            color = tower_color[tk]
            pygame.draw.circle(range_surf, (*color, 25), (cx, cy), int(radius*CELL))
            screen.blit(range_surf, (0, 0))

        # tower markers
        for tk in tower_list:
            pos, ci = tk
            cx, cy = pos[1]*CELL + CELL//2, pos[0]*CELL + CELL//2
            color = tower_color[tk]
            pygame.draw.circle(screen, color, (cx, cy), CELL//4)
            pygame.draw.circle(screen, (255, 255, 255), (cx, cy), CELL//4, 2)

        # right panel: tower legend with capacity bars
        panel_x = cols * CELL + 10
        y = 10
        header = font.render("TOWERS", True, (255, 255, 255))
        screen.blit(header, (panel_x, y))
        y += 25

        total_cost = 0
        for tk in tower_list:
            pos, ci = tk
            radius, cap, cst = tower_configs[ci]
            served = towers[tk]
            load = sum(demand[c[0]][c[1]] for c in served)
            total_cost += cst
            color = tower_color[tk]

            # tower info
            label = font_sm.render(f"{pos} R={radius} C={cap} $={cst}", True, color)
            screen.blit(label, (panel_x, y))
            y += 16

            # capacity bar
            bar_w = 140
            bar_h = 10
            fill = min(load / cap, 1.0)
            pygame.draw.rect(screen, (40, 40, 40), (panel_x, y, bar_w, bar_h))
            bar_color = color if fill < 0.9 else (255, 50, 50)
            pygame.draw.rect(screen, bar_color, (panel_x, y, int(bar_w * fill), bar_h))
            load_txt = font_sm.render(f"{load}/{cap}", True, (200, 200, 200))
            screen.blit(load_txt, (panel_x + bar_w + 5, y - 2))
            y += 20

        # total cost
        y += 10
        cost_txt = font.render(f"TOTAL: ${total_cost}", True, (255, 255, 100))
        screen.blit(cost_txt, (panel_x, y))

        pygame.display.flip()
        pygame.time.wait(50)

    pygame.quit()

In [41]:
rows, cols = 5, 5
demand = [
    [3, 1, 4, 1, 5],
    [9, 2, 6, 5, 3],
    [5, 8, 9, 7, 9],
    [3, 2, 3, 8, 4],
    [6, 2, 6, 4, 3],
]
tower_configs = [
    (1, 10, 12),
    (2, 20, 28),
    (3, 35, 45),
]

solution, cost = solve(rows, cols, demand, tower_configs)
print_solution(solution, demand, tower_configs)
run_viz(rows, cols, demand, solution, tower_configs)

  Tower (2, 2) cfg=2 R=3 C=35 $=45 | serves 4 cells, load=35/35
  Tower (2, 3) cfg=2 R=3 C=35 $=45 | serves 6 cells, load=35/35
  Tower (3, 0) cfg=0 R=1 C=10 $=12 | serves 3 cells, load=10/10
  Tower (0, 1) cfg=0 R=1 C=10 $=12 | serves 4 cells, load=10/10
  Tower (4, 0) cfg=0 R=1 C=10 $=12 | serves 2 cells, load=8/10
  Tower (2, 3) cfg=1 R=2 C=20 $=28 | serves 6 cells, load=20/20
